# 03 – Preprocessing: Manejo de Valores Faltantes

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Identificar, cuantificar y tratar los valores faltantes (missing values) del dataset mediante estrategias apropiadas para cada tipo de variable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ─── Cargar datos ─────────────────────────────────────────────────────────────
RAW_PATH = os.path.join('..', 'data', 'raw', 'epen_snapshot.csv')
try:
    df = pd.read_csv(RAW_PATH)
except FileNotFoundError:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'edad': np.where(np.random.rand(n) < 0.02, np.nan, np.random.randint(14, 70, n)).astype(float),
        'sexo': np.random.choice(['Hombre', 'Mujer'], n),
        'nivel_educativo': np.random.choice(
            ['Sin instrucción', 'Primaria', 'Secundaria', 'Preparatoria', 'Universidad', 'Posgrado'], n
        ),
        'estado_civil': np.random.choice(['Soltero', 'Casado', 'Unión libre', 'Divorciado', 'Viudo'], n),
        'ingreso_mensual': np.where(np.random.rand(n) < 0.05, np.nan, np.random.exponential(8000, n)).round(2),
        'horas_trabajadas': np.random.randint(0, 60, n),
        'tipo_empleo': np.random.choice(['Formal', 'Informal', np.nan], n, p=[0.45, 0.45, 0.10]),
        'sector': np.random.choice(['Agricultura', 'Industria', 'Comercio', 'Servicios', 'Gobierno'], n),
        'condicion_actividad': np.random.choice(
            ['Ocupado', 'Desocupado', 'No PEA'], n, p=[0.60, 0.10, 0.30]
        ),
    })

df['target_desocupado'] = (df['condicion_actividad'] == 'Desocupado').astype(int)
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')

## 1. Diagnóstico de valores faltantes

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (df.isnull().mean() * 100).round(2).sort_values(ascending=False)
diag = pd.DataFrame({'Nulos': missing, 'Nulos_%': missing_pct})
print(diag)

# Visualización
diag_plot = diag[diag['Nulos'] > 0]
if not diag_plot.empty:
    diag_plot['Nulos_%'].plot(kind='barh', color='salmon', edgecolor='black', figsize=(8, 4))
    plt.title('Porcentaje de valores faltantes por variable')
    plt.xlabel('% Faltantes')
    plt.tight_layout()
    plt.show()
else:
    print('No hay valores faltantes en el dataset.')

## 2. Estrategias de imputación

| Variable | Tipo | Estrategia |
|---|---|---|
| `edad` | Numérica | Imputar con la **mediana** |
| `ingreso_mensual` | Numérica | Imputar con la **mediana** |
| `tipo_empleo` | Categórica | Imputar con categoría **'Sin empleo'** (MNAR: Missing Not At Random) |

In [ ]:
df_clean = df.copy()

# Variables numéricas → mediana
for col in ['edad', 'ingreso_mensual']:
    if col in df_clean.columns and df_clean[col].isnull().any():
        mediana = df_clean[col].median()
        df_clean[col].fillna(mediana, inplace=True)
        print(f'{col}: imputado con mediana = {mediana:.2f}')

# tipo_empleo → categoría especial
if 'tipo_empleo' in df_clean.columns:
    df_clean['tipo_empleo'].fillna('Sin empleo', inplace=True)
    print('tipo_empleo: imputado con "Sin empleo"')

print(f'\nFaltantes restantes: {df_clean.isnull().sum().sum()}')

In [ ]:
# Guardar dataset preprocesado (paso 1)
os.makedirs(os.path.join('..', 'data', 'processed'), exist_ok=True)
df_clean.to_csv(os.path.join('..', 'data', 'processed', 'epen_missing_handled.csv'), index=False)
print('Dataset guardado: epen_missing_handled.csv')